In [ ]:
import MotoricaInterface as mi; assert(mi.__version__=="0.0.8") # last time launched
from MotoricaInterface import SerialInterface_2EMG
import numpy as np
import time
import pandas as pd

# GUI select

In [44]:
df_ports = mi.utils.list_devices()
df_ports

,device,name,description,hwid,vid,pid,serial_number,location,manufacturer,product,interface
0,/dev/cu.BLTH,cu.BLTH,n/a,n/a,NaN,NaN,None,None,None,None,None
1,/dev/cu.ESP32-BT-Slave,cu.ESP32-BT-Slave,n/a,n/a,NaN,NaN,None,None,None,None,None
2,/dev/cu.BoseQC35II,cu.BoseQC35II,n/a,n/a,NaN,NaN,None,None,None,None,None
3,/dev/cu.Bluetooth-Incoming-Port,cu.Bluetooth-Incoming-Port,n/a,n/a,NaN,NaN,None,None,None,None,None
4,/dev/cu.usbserial-14420,cu.usbserial-14420,USB Serial,USB VID:PID=1A86:7523 LOCATION=20-4.2,6790.0,29987.0,None,20-4.2,None,USB Serial,None


In [45]:
import ipywidgets as widgets
from IPython.display import display

# Create the dropdown widget
style = {'description_width': 'initial'}
dropdown = widgets.Dropdown(
    options=df_ports.device.unique(),
    description='Select Device:',
    disabled=False,
    style=style
)

# Create a message field
message = widgets.Label(value="")

ser = None
def select(b):
    global ser
    ser = SerialInterface_2EMG(dropdown.value, 
                                  debug=False, 
                                  logging_enabled=False)
    message.value = f"Selected: {dropdown.value}\n{ser}"
    

# Create the button widget
button = widgets.Button(
    description='Launch Device',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Launch the device'
)

# Link the button to the select function
button.on_click( select)

# Use HBox to align the dropdown and button horizontally
hbox = widgets.HBox([dropdown, button])

# Use VBox to stack the HBox and the message field vertically
vbox = widgets.VBox([hbox, message])

# Display the VBox containing the HBox and message field
display(vbox)

# Start parsing

## Override


In [2]:
ser = SerialInterface_2EMG('/dev/cu.usbserial-14420', 
                            debug=False, 
                            logging_enabled=False, 
                            timeout=0.5)


In [3]:
ser.start()

Start Serial Reading


## Testing interfaces

In [4]:
ser.DEBUG=False

print('\r\nReading Gains')
time.sleep(0.1)
# ser.flush_rx_packs()
print(ser.ReadGain(0x01, timeout=0.2))
# ser.flush_rx_packs()
print(ser.ReadGain(0x02, timeout=0.2))

print('\r\nSetting Gains')
ser.WriteGain(0x01, 0);   print(ser.gain_table[255-0])
ser.WriteGain(0x02, 240); print(ser.gain_table[255-240])

print('\r\nReading Gains')
time.sleep(0.1)
# ser.flush_rx_packs()
print(ser.ReadGain(0x01, timeout=0.2))
# ser.flush_rx_packs()
print(ser.ReadGain(0x02, timeout=0.2))

print('\r\nReading Version')
# ser.flush_rx_packs()
print(ser.ReadVersion(timeout=0.5))

print('\r\nReading Select Result Type')
# ser.flush_rx_packs()
print(ser.ReadSelectResultType(0x01, timeout=.5))
# ser.flush_rx_packs()
print(ser.ReadSelectResultType(0x02, timeout=.5))



Reading Gains
400.0
30.142254

Setting Gains
400.0
1.384083

Reading Gains
400.0
1.384083

Reading Version
6

Reading Select Result Type
8
8


# Simple Pack Counter

This code cell initializes the serial interface for reading EMG data packets.  
It sets the debug mode to False, initializes the packet counter and timestamps,  
and flushes any existing packets in the receive buffer.  
The main loop continuously checks for new packets in the receive buffer.  
If a packet is found, it prints the packet details including the packet header and timestamp difference.  
If no packet is found, it sleeps for a short duration before checking again.  

In [8]:
ser.DEBUG = False
pack_counter = 0;
t_start = time.time()
t_last_pack = 0;

global while_True
while_True = True

ser.flush_rx_packs()
from ipywidgets import Button
from IPython.display import display
import threading

def stop_loop(b):
    global while_True
    while_True = False

stop_button = Button(description="Stop Loop")
stop_button.on_click(stop_loop)
display(stop_button)

def loop():
    global while_True
    global pack_counter
    global t_start
    global t_last_pack
    while while_True:
        if (len(ser.rx_pack)):
            now = time.time()
            ts, RX = ser.rx_pack.pop()
            print(f'{while_True} N={pack_counter}\t'
                  f'From thread {ts-ser.thread_start_time:.3f}\t'
                  f'From main {time.time() - t_start:.2f}\t'
                  f'PACK_HEADER {RX[:2]}\t'
                  f'TS_DIFF={(now - t_last_pack)*1000:.0f}', 
                  end=' ' * 20 + '\r')
            t_last_pack = now
            pack_counter+=1
        else:
            time.sleep(0.01)
    
loop_thread = threading.Thread(target=loop)
loop_thread.start()


Button(description='Stop Loop', style=ButtonStyle())

# EMG Pack reader

In [4]:
ser.DEBUG = False
pack_counter = 0;
t_start = time.time()
t_last_pack = 0;
global while_True
while_True = True

ser.flush_rx_packs()

import socket

# Plotting to teleplot
teleplotAddr = ("127.0.0.1",47269)
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
def sendTelemetry(name, value):
    now = time.time() * 1000
    msg = name+":"+str(now)+":"+str(value)+"|g"
    sock.sendto(msg.encode(), teleplotAddr)
# / Plotting to teleplot    

from ipywidgets import Button
from IPython.display import display
import threading

def stop_loop(b):
    global while_True
    while_True = False

stop_button = Button(description="Stop Loop")
stop_button.on_click(stop_loop)
display(stop_button)

def loop():
    global while_True
    global pack_counter
    global t_start
    global t_last_pack
    global ser

    ser.flush_rx_packs()
    while_True = True
    while while_True:
        if (len(ser.rx_pack)):
            now = time.time()
            ts, RX = ser.rx_pack.pop()
            t_last_pack = now
            pack_counter+=1

            if (RX[0] == 0x01) & (RX[1] == 0x20):
                print(f'{pack_counter}\t{ time.time() - t_start:.2f}'
                      f'\t{len(ser.rx_pack)}'
                      f'\tEMG_A {RX[3]:03d}\tEMG_B {RX[4]:03d}', 
                      end = ' '*20 + '\r')
                sendTelemetry('emg1', RX[3])
                sendTelemetry('emg2', RX[4])
                msg = 'emg'+":"+str(RX[3])+":"+str(RX[4])+"|xy"
                sock.sendto(msg.encode(), teleplotAddr)
        else:
            time.sleep(0.01)
    
loop_thread = threading.Thread(target=loop)
loop_thread.start()


Button(description='Stop Loop', style=ButtonStyle())

Exception in thread Thread-4 (parse):
Traceback (most recent call last):
  File "/usr/local/anaconda3/envs/opto3/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/local/anaconda3/envs/opto3/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/mike/src/EMGSoftware/PythonScripts/virtual_phantom/MotoricaInterface/motorica_serial_interface.py", line 153, in parse
    RX[1 + 1 + 1 + RX_IDX] = ch
    ~~^^^^^^^^^^^^^^^^^^^^
IndexError: index 16 is out of bounds for axis 0 with size 16


# AI Gesture State Reader

In [51]:
import socket

# Plotting to teleplot
teleplotAddr = ("127.0.0.1",47269)
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
def sendTelemetry(name, value):
    now = time.time() * 1000
    msg = name+":"+str(now)+":"+str(value)+"|g"
    sock.sendto(msg.encode(), teleplotAddr)
# / Plotting to teleplot    

ser.DEBUG = False
pack_counter = 0;
t_start = time.time()

states = ["Neutral", 'RIGHT', 'LEFT']

ser.flush_rx_packs()
while True:
    if (len(ser.rx_pack)):
        ts, RX = ser.rx_pack.pop()
        pack_counter += 1
        if (RX[0] == 0x01) & (RX[1] == 0x83):
            print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tEMG STATE {RX[3]:01d} {states[RX[3]]}', end = ' '*20 + '\r')
            sendTelemetry('STATE', RX[3])
            # print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tEMG STATE {RX[3]:01d} {states[RX[3]]}')

    else:
        time.sleep(0.01) 

KeyboardInterrupt: 

# Test bench for training AI

In [5]:
ser.DEBUG = False
pack_counter = 0;
t_start = time.time()
t_last_pack = 0;
global while_True
while_True = True

ser.flush_rx_packs()

import socket

# Plotting to teleplot
teleplotAddr = ("127.0.0.1",47269)
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
def sendTelemetry(name, value):
    now = time.time() * 1000
    msg = name+":"+str(now)+":"+str(value)+"|g"
    sock.sendto(msg.encode(), teleplotAddr)
# / Plotting to teleplot    

from ipywidgets import Button
from IPython.display import display
import threading

def stop_loop(b):
    global while_True
    while_True = False

stop_button = Button(description="Stop Loop")
stop_button.on_click(stop_loop)
display(stop_button)

def loop():
    global while_True
    global pack_counter
    global t_start
    global t_last_pack
    global ser

    ser.flush_rx_packs()
    while_True = True

    states = ["Neutral", 'RIGHT', 'LEFT']
    while while_True:
        if (len(ser.rx_pack)):
            now = time.time()
            ts, RX = ser.rx_pack.pop()
            t_last_pack = now
            pack_counter+=1

            if (RX[0] == 0x01) & (RX[1] == 0x20):
                print(f'{pack_counter}\t{ time.time() - t_start:.2f}'
                        f'\t{len(ser.rx_pack)}'
                        f'\tEMG_A {RX[3]:03d}\tEMG_B {RX[4]:03d}', 
                        end = ' '*20 + '\r')
                sendTelemetry('emg1', RX[3])
                sendTelemetry('emg2', RX[4])


            if (RX[0] == 0x01) & (RX[1] == 0x83):
                print(f'{pack_counter}\t{ time.time() - t_start:.2f}\tEMG STATE {RX[3]:01d} {states[RX[3]]}', end = ' '*20 + '\r')
                sendTelemetry('STATE', RX[3])

        else:
            time.sleep(0.01)
    
loop_thread = threading.Thread(target=loop)
loop_thread.start()

time.sleep(5)
ser.WriteProtocolFlag(0x01)


Button(description='Stop Loop', style=ButtonStyle())

Sending bytearray(b'\x01\x82\x01\x018')         


In [10]:
import socket

# Plotting to teleplot
teleplotAddr = ("127.0.0.1",47269)
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
def sendTelemetry(name, value):
    now = time.time() * 1000
    msg = name+":"+str(now)+":"+str(value)+"|g"
    sock.sendto(msg.encode(), teleplotAddr)
# / Plotting to teleplot    

ser.DEBUG = False
pack_counter = 0;
t_start = time.time()



ser.flush_rx_packs()
while True:
    if (len(ser.rx_pack)):
        ts, RX = ser.rx_pack.pop()
        pack_counter += 1

        
    else:
        time.sleep(0.01) 

KeyboardInterrupt: 

In [4]:
print('\r\nSetting Gains')
ser.WriteGain(0x01, 230); 
ser.WriteGain(0x02, 230); 

print('\r\nReading Gains')
time.sleep(0.1)
# ser.flush_rx_packs()
print(ser.ReadGain(0x01, timeout=0.2))
# ser.flush_rx_packs()
print(ser.ReadGain(0x02, timeout=0.2))


Setting Gains

Reading Gains
3.844675
3.844675
